# Section 2 — RAG Pipeline (Gemini)
**Setup:** Add `GOOGLE_API_KEY` to Colab Secrets.

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community faiss-cpu

In [ ]:
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
print('Key loaded.')

In [ ]:
os.makedirs('documents', exist_ok=True)
docs = {
    'doc1_refund_policy.md': '# Refund Policy\n\n## Eligibility\n- Orders 30+ min late: full refund\n- Missing items: partial refund (cost + 10% credit)\n- Wrong items: full replacement or refund\n\n## Process\n1. Request within 24 hours\n2. Reviewed in 2 business days\n3. Refund in 3-5 days\n\n## Exceptions\n- Cancel after prep: 20% fee\n- Wrong address: no refund',
    'doc2_delivery_zones.md': '# Delivery Zones\n\n## Zone A (Maadi, Zamalek, Downtown)\n- Time: 20-30 min\n- Fee: 15 EGP\n- Hours: 8AM-2AM\n- Surge +5 EGP peak hours\n\n## Zone B (Heliopolis, Nasr City)\n- Time: 35-50 min\n- Fee: 25 EGP\n- Hours: 9AM-12AM\n- Min order: 100 EGP\n\n## Zone C (New Cairo, Sheikh Zayed)\n- Time: 45-70 min\n- Fee: 40 EGP\n- Hours: 10AM-11PM',
    'doc3_partners.md': '# Partners\n\n## Commission\n- Standard: 18%\n- Premium (4.5+): 15%\n- New (3 months): 12%\n\n## Quality\n- Min rating: 3.8/5\n- Below 3.0 for 60 days: removed\n\n## Payments\n- Weekly Monday settlements',
}
for name, content in docs.items():
    with open(f'documents/{name}', 'w') as f: f.write(content)
print(f'{len(docs)} docs created')

In [ ]:
import glob
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate

documents = []
for fp in glob.glob('documents/*.md'):
    with open(fp) as f:
        documents.append(Document(page_content=f.read(), metadata={'source': os.path.basename(fp)}))

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=['\n## ','\n\n','\n','. '])
chunks = splitter.split_documents(documents)
for i, c in enumerate(chunks): c.metadata['chunk_id'] = i
print(f'{len(chunks)} chunks')

emb = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004', google_api_key=os.environ['GOOGLE_API_KEY'])
vectorstore = FAISS.from_documents(chunks, emb)
print('FAISS built')

In [ ]:
PROMPT = '''Answer ONLY from context. If not enough info say:
"I don't have that info. Contact support@quickbite.com"
Cite the document name.

Context:\n{context}\n\nQuestion: {question}\n\nAnswer:'''

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0, google_api_key=os.environ['GOOGLE_API_KEY'])
prompt = ChatPromptTemplate.from_template(PROMPT)

def ask(question):
    results = vectorstore.similarity_search_with_score(question, k=3)
    relevant = [(d, 1/(1+s)) for d, s in results if 1/(1+s) >= 0.3]
    if relevant:
        context = '\n---\n'.join(f'[{d.metadata["source"]}] {d.page_content}' for d,_ in relevant)
        sources = [d.metadata['source'] for d,_ in relevant]
    else:
        context = 'NO RELEVANT CONTEXT.'
        sources = []
    resp = llm.invoke(prompt.format(context=context, question=question))
    print(f'\nQ: {question}\nA: {resp.content}\nSources: {sources}\n')

print('RAG ready.')

In [ ]:
print('='*50, '\n  Q1: Refund case\n', '='*50)
ask('My order came 45 minutes late. Can I get a refund?')

print('='*50, '\n  Q2: Zone info\n', '='*50)
ask('Delivery fee and hours for Maadi?')

print('='*50, '\n  Q3: No answer (hallucination test)\n', '='*50)
ask('What programming language is the app built with?')